# Does El Niño really bring rain to California?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT5_el_nino_and_the_rivers.ipynb).

Every few years the trade winds slacken and a band of the equatorial Pacific warms by a degree or
two. That is El Niño, and NOAA has been measuring it monthly since 1948 as a sea
surface temperature anomaly in a box on the equator called Niño 3.4. California's winter storm
track is supposed to move with it, and by December of a warm year the newspapers say so: *this is
the year the drought breaks.*

Rivers are how you check. The USGS has been reading California stream gauges since before anyone
measured El Niño — 192 of them, in this notebook's list, with 50 years or more
of daily flow and still reporting. The oldest holds 136 years.

So the claim is testable with two files and one number: how tightly does a winter's Pacific
temperature move with the water that came down a river that year? The answer turns out to depend on
which river you ask — and on a choice about the calendar that nobody in the newspaper story ever
makes.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces one result — that a river in southern California
does move with El Niño — and then stops helping. From there on every section is a sentence
describing what to find out and an empty cell to find it out in. There is no worked example above
to pattern-match against, because on a real question there never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a result
you can get two ways, a number you can predict before you compute it, and a claim you can try to
break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say whether El Niño measurably changes how much water comes down a California
river, put an interval on the answer, and say where in the state the answer changes — and then say
what the data cannot tell you about where the boundary is.

**The skills.** Turn a daily record into a yearly one, which means choosing where a year starts and
finding out what that choice costs. Compare two correlations honestly, which is not the same as
computing two of them. Resample the thing that is actually paired.

**The four questions, in order:**

1. Does El Niño show up in one California river?
2. Does the answer change if you go north?
3. Which twelve months are a year?
4. Is the north–south gap real, or two noisy numbers?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

Three live archives, each read straight from its source with a copy stored with the course
behind it. (The map also reads `coastlines.csv`, which ships with the course and has no upstream to
be live from, so it is read directly.)

- **NOAA PSL** publishes the Niño 3.4 anomaly as a plain text table: one row per year, twelve
  columns, and then a few lines of notes signed at the bottom. Its "no reading" value is
  **`-99.99`**, which appears both in this year's unmeasured months and, once, on a line of its
  own — so reading it as a missing value clears the notes as well as the gaps.
- **USGS** publishes daily discharge in a format called RDB, which carries three traps in one
  read. There is a format row (`5s 15s 20d 14n`) under the header that is not data; the discharge
  column is named after an internal timeseries id, so it is called something different for every
  gauge and must be found by its `_00060_00003` ending; and the discharge column has to be
  forced to numbers, because USGS writes a word (`Ice`, `Ssn`) where a reading is frozen or
  seasonal. Neither of the two gauges worked below needed that last one — but a gauge that
  freezes will, and you will not be told.
- **The USGS site inventory** for California, which is where the gauge list comes from. It is the
  same RDB format, one row per site per measurement it publishes, with the first and last day of
  each record and how many days are in it. The first section builds the list from it in front of
  you, rather than handing you a finished file, because the filter that turns
  2,420 sites into a shortlist is a judgement and you should be able to argue with
  it.

The discharge query is pinned to end on **2025-09-30**, the last day of water year
2025. Without that it would grow by a row every morning and quietly change every number
below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name, **options):
    """Read one live source; fall back to the copy stored with the course."""
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv(url, **options)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name, **options)


NINO = "https://psl.noaa.gov/data/correlation/nina34.anom.data"
GAUGE = ("https://waterservices.usgs.gov/nwis/dv/?format=rdb&parameterCd=00060"
         "&statCd=00003&startDT=1900-01-01&endDT=2025-09-30&sites=")
SITES = ("https://waterservices.usgs.gov/nwis/site/?format=rdb&stateCd=ca"
         "&parameterCd=00060&outputDataTypeCd=dv&seriesCatalogOutput=true&siteStatus=all")

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
FULL_YEAR = 350      # days a twelve-month window needs before it counts as a year

# --- El Nino, once. One row per year, twelve columns, then a few lines of notes. ---
nino_rows = load(NINO, "trackT5_nina34.txt", sep=r"\s+", skiprows=1, header=None,
                 names=["year"] + MONTHS, na_values="-99.99")
nino = nino_rows.apply(pd.to_numeric, errors="coerce").dropna(subset=["year"])

# El Nino is a WINTER thing, and the winter before a California spring is December of the year
# before plus January and February of this one. shift(1) reaches back one row for that December.
winter = pd.DataFrame({"year": nino["year"].astype(int),
                       "djf": (nino["Dec"].shift(1) + nino["Jan"] + nino["Feb"]) / 3}).dropna()


def flow(site):
    """Every daily reading from one USGS gauge: one row per day, discharge in cubic feet/second."""
    # 1. One gauge's whole daily record. `load` asks the USGS server first and falls back to the
    #    copy stored with the course, so this cell still runs when the network does not.
    raw = load(GAUGE + site, "trackT5_dv_" + site + ".tsv.gz",
               sep="\t", comment="#", low_memory=False)
    raw = raw[raw["agency_cd"] == "USGS"]        # drops the "5s 15s 20d 14n" format row
    for name in raw.columns:                     # the discharge column carries an internal id,
        if name.endswith("_00060_00003"):        # so it is named differently at every gauge
            column = name
    # 2. Keep the two columns this notebook needs. A day the gauge did not report arrives as text
    #    rather than a number, so `errors="coerce"` blanks it and `dropna` drops the row — a day
    #    with no reading must not be averaged in as a day of no water.
    return pd.DataFrame({"date": pd.to_datetime(raw["datetime"]),
                         "cfs": pd.to_numeric(raw[column], errors="coerce")}).dropna()


def water_year(dates, start_month):
    """Which twelve-month year each date falls in, if the year begins in `start_month`.

    A year is named for the calendar year it ENDS in, so 1 October 1999 is in water year 2000.
    """
    shift = (13 - start_month) % 12
    return dates.dt.year + (dates.dt.month - 1 + shift) // 12


def paired(site, start_month):
    """One row per year: the winter Nino 3.4 index, and that year's mean flow at one gauge."""
    # 1. Label every day with the twelve-month year it falls in, then average the days inside each
    #    of those years. `start_month` is the choice this whole notebook turns on: move it and the
    #    boundary between one year and the next moves with it.
    daily = flow(site)
    daily["year"] = water_year(daily["date"], start_month)
    per_year = daily.groupby("year")["cfs"]
    # 2. Carry `days` next to the mean — how many daily readings that year actually got — because
    #    the mean of a year the gauge only half-covered is not the mean of a year.
    yearly = pd.DataFrame({"year": per_year.mean().index,
                           "cfs": per_year.mean().values,
                           "days": per_year.size().values})
    whole = yearly[yearly["days"] >= FULL_YEAR]     # a year missing a season is not a year
    return whole.merge(winter, on="year")           # and only years the Nino record reaches


def correlation(a, b):
    """How tightly two columns move together: +1 rises together, -1 opposite, 0 not at all."""
    return np.corrcoef(a, b)[0, 1]


coast = pd.read_csv(CACHE + "/coastlines.csv")

print("winters with a Nino index:", len(winter), "—", winter["year"].min(), "to",
      winter["year"].max())

## Does El Niño show up in one California river?

California has 2,420 sites with a daily-mean discharge record, and most of them are
no use here: a gauge with fifteen years of data cannot say anything about a cycle that turns up
every few years, and a gauge measuring an irrigation canal is measuring a decision somebody made,
not weather. Two filters cut it down. The first is arithmetic — 50+ years of readings, at
least 95% of the days present, and still reporting in 2020 or later. The second is not: it reads
the station's **name** and throws it out if the name advertises plumbing.

In [ ]:
inventory = load(SITES, "trackT5_ca_inventory.tsv.gz", sep="\t", comment="#", low_memory=False)
discharge = inventory[(inventory["agency_cd"] == "USGS")
                      & (inventory["parm_cd"].astype(str) == "00060")     # discharge
                      & (inventory["stat_cd"].astype(str) == "00003")     # daily mean
                      & (inventory["data_type_cd"] == "dv")]

days = pd.to_numeric(discharge["count_nu"], errors="coerce")
last = pd.to_datetime(discharge["end_date"], errors="coerce")
span = (last - pd.to_datetime(discharge["begin_date"], errors="coerce")).dt.days + 1
long_record = discharge[(days >= 50 * 365) & (days / span >= 0.95)
                        & (last.dt.year >= 2020)]

# The line most worth arguing with in this notebook. A canal, a spill, a conduit or a gauge
# reading BL (below) a dam is measuring an operating decision, so it goes — but only its NAME
# says so, and a name is not a measurement.
BUILT = r"CANAL|\bCN\b|AQUEDUCT|DRAIN|SPILL|WASTEWAY|CONDUIT|TUNNEL|FLUME|INTAKE|POWERPLANT|\bPP\b|OUTLET|DIVERSION"
DAMMED = r"\bBL\b|\bBLW\b|BELOW|\bDAM\b"
station = long_record["station_nm"].str.upper()
natural = long_record[~station.str.contains(BUILT) & ~station.str.contains(DAMMED)]

gauges = pd.DataFrame({"site_no": natural["site_no"].astype(str).str.zfill(8).values,
                       "station_nm": natural["station_nm"].values,
                       "lat": pd.to_numeric(natural["dec_lat_va"], errors="coerce").values,
                       "lon": pd.to_numeric(natural["dec_long_va"], errors="coerce").values})
gauges = gauges.sort_values("lat", ascending=False)

print("California discharge sites:", discharge["site_no"].nunique())
print("with a long, complete, current record:", len(long_record))
print("of those, not named as built or below a dam:", len(gauges))

Here is what survives, on a map. Every dot is a gauge with at least 50 years of daily
discharge; 97 of them are north of 38°N, 36 between 36 and 38, and
59 south of 36. The two marked in red are the ones this notebook works with, and
one of them will be yours to replace.

In [ ]:
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.scatter(gauges["lon"], gauges["lat"], s=8, color="0.25")
demo = gauges[gauges["site_no"].isin(["11152000", "11477000"])]
plt.scatter(demo["lon"], demo["lat"], s=60, color="firebrick", marker="v")
plt.xlim(-125, -113.5)
plt.ylim(32, 42.5)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (°E)")
plt.ylabel("latitude (°N)")
plt.title(f"{len(gauges)} California gauges with 50+ years of daily discharge")
plt.show()

Take the southern one first: **ARROYO SECO NR SOLEDAD CA**, at 36.28°N, reading since
1901-10-01. Before pairing anything with anything, look at what a Californian river does in
an ordinary year.

In [ ]:
south_daily = flow("11152000")
south_daily["month"] = south_daily["date"].dt.month

by_month = south_daily.groupby("month")["cfs"].mean()

plt.bar(by_month.index, by_month.values, color="0.4")
plt.xlabel("calendar month (1 = January)")
plt.ylabel("mean discharge (cubic feet per second)")
plt.title(f"ARROYO SECO NR SOLEDAD CA: the average year ({len(south_daily)} daily readings)")
plt.locator_params(axis="x", integer=True)
plt.show()

print("wettest month:", MONTHS[by_month.idxmax() - 1], " driest:", MONTHS[by_month.idxmin() - 1])

That is the whole reason a hydrologist does not use the calendar. Flow peaks in
Feb at about 551 cubic feet per second and bottoms out in
Sep near 5 — a factor of
117. A **water year** therefore runs from 1 October to
30 September: it starts in the dry gap, so one winter's storms stay inside one year instead of
being cut in half by 31 December.

`paired(site, start_month)` does the rest — it fetches a gauge, labels every day with the year that
`start_month` puts it in, averages each year that has at least 350 days of readings, and
lines the result up against the winter Niño index. There is no default for `start_month`: you have
to write the choice down every time, because it is a choice.

In [ ]:
south = paired("11152000", 10)

print("paired years:", len(south), "—", south["year"].min(), "to", south["year"].max())
print(south.head())

In [ ]:
assert winter["year"].min() == 1949, \
    "the winter index should start in 1949 — its first December is the year before"
assert 70 <= len(south) <= 80, \
    "expected about 77 paired water years; a very different number means the day "\
    "count filter or the Nino join dropped more than it should"
assert south["cfs"].min() > 0, "a year's mean discharge cannot be zero or negative"
print(f"✓ the data — {len(winter)} winters with a Niño index, {len(south)} complete water "
      f"years at ARROYO SECO NR SOLEDAD CA")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the files are the files, the join is the join, the numbers below
are the numbers. Everything from here is yours, and the safety net is gone — nothing will tell you
when you have it right.

Now the actual question, on one gauge. One dot per water year: the winter's Niño 3.4 anomaly across
the bottom, that year's mean discharge up the side.

`correlation` turns the whole cloud into one number. It is **+1** if the dots lie exactly on a
rising line, **0** if the cloud has no tilt at all, **−1** if it falls — and it is the closest
thing to a single answer this question has.

In [ ]:
r_south = correlation(south["djf"], south["cfs"])

plt.scatter(south["djf"], south["cfs"], s=18, color="0.3")
plt.axvline(0, color="0.7", lw=1)
plt.xlabel("winter (Dec–Feb) Niño 3.4 anomaly (°C)")
plt.ylabel("water-year mean discharge (cubic feet per second)")
plt.title(f"ARROYO SECO NR SOLEDAD CA, {len(south)} water years — r = {r_south:.3f}")
plt.show()

print("correlation:", round(r_south, 3))

**r = +0.309** over 77 water years. A warm Pacific winter and a wet year in
the Arroyo Seco do go together, and the newspapers are not making it
up.

Notice what the figure does *not* show, though. Nothing about that cloud of 77 dots
announces +0.309 rather than +0.087 or +0.50. You would
not read that number off the picture, which is worth remembering for the rest of the notebook: from
here on the eye is no help and the arithmetic is all you have.

### Predict before you run

EEL R A SCOTIA CA drains the far north coast, 4.2 degrees of
latitude — about 467 km — up the state. Its correlation with the same winter index is the very next
thing you will compute. Write down what you think it is first. Change `my_guess` and run the cell.

A wrong guess you committed to is worth more than a right answer you were shown.

In [ ]:
my_guess = None    # ← your number, written down before you look

In [ ]:
assert my_guess is not None, \
    "write a number into my_guess in the cell above — the commitment is the point, "\
    "and a guess you made before you saw the answer is the only one that can teach you anything"
print("✓ committed — I think", my_guess, "is the northern river's correlation with winter Niño 3.4")

## Does the answer change if you go north?

EEL R A SCOTIA CA is site `11477000`, at 40.49°N. It has been reading since
1910-10-01 and it is on the map above, the northern red triangle.

### ✏️ Your turn 1

Run the same pipeline on `"11477000"` with the same water year, draw the same scatter, and print its
correlation next to the southern one so both are on the page.

Then print one more line answering it in a sentence, on your own two numbers: does El Niño bring
rain to California, or does it bring rain to *part* of California — and what would you have
concluded if this notebook had handed you the northern gauge first?

In [ ]:
# ← your answer here



A correlation is already a modelling choice: it assumes the relationship is a straight line, and it
turns 77 pairs into one number that nobody can check by eye. Before trusting it, it is
worth asking what the dumbest possible version of this analysis says.

**Baseline:** Write the dumbest rule you can, first. Any model that cannot beat it is decoration.

### ✏️ Your turn 2

The dumbest version of this question needs no correlation at all: **split the years into three
piles and take the average of each.** NOAA calls a winter El Niño when the Niño 3.4 anomaly is at
or above +0.5 °C and La Niña at or below −0.5, with everything between called neutral.

For **both** gauges, print how many years fall in each pile, the mean discharge of each, and the
El Niño mean divided by the La Niña mean. Draw whatever figure makes the two gauges comparable
despite one of them carrying about fifty times more water than the other.

Then print one more line answering it: which of the two numbers — the correlation from Your turn 1
or this ratio — would you put in a newspaper, and what does the other one know that it does not?

In [ ]:
# ← your answer here



## Which twelve months are a year?

Every number so far rests on a decision made in the setup cell and never argued for: that a year
starts on 1 October. The reason was in the figure of the average year — October is the dry gap, so
a water year keeps one winter's storms together.

But *dry gap* is a judgement, not a boundary. September would do. November would do. And a
calendar year, which is what you get if you reach for `date.dt.year` without thinking about it, is
what almost every beginner uses. `paired` takes `start_month` precisely so you can try them.

This is the one real decision in this track. Make it, and report what it cost.

### ✏️ Your turn 3

Run **both** gauges under four twelve-month windows: starting in October, November, September, and
January. `paired(site, 1)` is the calendar year.

For each window print how many paired years survive and the two correlations, and print the
**gap** between them — the southern correlation minus the northern one — because the gap is the
claim this track is actually making.

Then print one more line answering it, and answer it by reading **down** the two correlation
columns rather than across the gap column. One window disagrees with the other three — but when it
disagrees, only one of the two rivers has moved. Say which river moves and which sits still, and
say what that does to the gap you would put in a headline.

In [ ]:
# ← your answer here



### ✏️ Your turn 4

Two or three paragraphs, quoting **your own four rows** — both correlation columns, not just the
gap.

1. Which window would you report, and what does the choice cost? Say what a reader loses by not
   being shown the other three.
2. The calendar year is not a mistake — plenty of published work reports calendar-year runoff, and
   nothing in the data says it is wrong. Your four gaps are therefore not four attempts at one
   number. One river is responsible for all of the disagreement between them: say which, and say
   what that does to the headline you would write. If the gap you would report is the biggest of
   the four, what does a reader have to be told for that headline to be honest?
3. You cannot yet say *why* one river moved and the other did not, but you can say what you would
   have to know. Name it.

*(Double-click this cell and replace this line with your answer.)*

### Why the northern river moved and the southern one did not

Your own four rows say the calendar year moves the north and leaves the south where it was. They
cannot say why, and the why is worth three more lines of code, because it decides how much of the
gap to believe.

The two conventions agree about nine months of the year and disagree about one quarter of it:
**October to December**. They also disagree about *which* October to December. The water year takes
the autumn that runs up to and into a winter — and December of that autumn is one of the three
months the Niño index is built from. The calendar year takes the autumn that comes nine months
*after* that winter ended.

So there are three pieces to look at, at both gauges: the winter itself, and the two rival autumns.

In [ ]:
def piece(site, months, year_of):
    """Mean flow over `months` of a calendar year, lined up with the winter it belongs to.

    year_of = 0 when those months come after the winter inside the same calendar year;
    year_of = 1 when the winter is the one they run into, so Oct-Dec 1982 serves winter 1983.
    """
    # 1. Keep the months asked for, and average them inside each CALENDAR year — the grouping is
    #    on the date's own year, before any relabelling.
    daily = flow(site)
    monthly = daily[daily["date"].dt.month.isin(months)]
    mean = monthly.groupby(monthly["date"].dt.year)["cfs"].mean()
    # 2. `+ year_of` is the relabelling: it slides those calendar years onto the winter they
    #    serve, so the merge lines each average up against the Nino index it should be compared to.
    return pd.DataFrame({"year": mean.index + year_of,
                         "cfs": mean.values}).merge(winter, on="year")


for label, months, year_of in [("Jan-Mar, the winter itself      ", [1, 2, 3], 0),
                               ("Oct-Dec BEFORE it (water year)  ", [10, 11, 12], 1),
                               ("Oct-Dec AFTER it (calendar year)", [10, 11, 12], 0)]:
    n = piece("11477000", months, year_of)
    s = piece("11152000", months, year_of)
    print(f"{label}   north {correlation(n['djf'], n['cfs']):+.3f}"
          f"   south {correlation(s['djf'], s['cfs']):+.3f}")

for site, label in [("11477000", "north"), ("11152000", "south")]:
    by_month = flow(site).groupby(flow(site)["date"].dt.month)["cfs"].mean()
    share = 100 * by_month.loc[[10, 11, 12]].sum() / by_month.sum()
    print(f"{label}: Oct-Dec is {share:.0f}% of the average year's water")

Those five lines are the whole explanation of the 3.6× you found, and of why
the southern column never moved.

The northern river's winter and its two autumns do not agree with each other. Its January–March
flow goes with El Niño at +0.224. The autumn running **into** that
same winter goes the other way, -0.191 — an El Niño autumn on the
north coast is, if anything, dry. The autumn nine months **after** it comes out at
+0.232. And Oct–Dec is
22% of the Eel's water, so which of those two autumns your year contains is
not a rounding term: the water year averages a positive winter against a negative autumn and lands
low, and the calendar year swaps in an autumn that does not pull the other way.

At the southern gauge the same three months are only 11% of the year's
water and both autumns come out near zero
(-0.031 and -0.032), so the
choice has almost nothing to work with. That is the whole asymmetry.

**One warning before you use this.** It is tempting to read
+0.232 as a second sighting of the same El Niño and conclude that
the calendar year is the better estimate. It is not, and the index says so itself: the Niño 3.4
autumn that runs into a winter is that winter over again
(r = +0.966 between the two indices), while the autumn after it has no memory
of it at all (r = +0.004). Whatever the calendar year's autumn is contributing
to the northern correlation, it is not that winter's El Niño. So neither convention gives the
honest answer: each one staples three months of unrelated water onto a number about winter.

Which leaves an obvious question the four windows never asked: what does the correlation look like
on the months the Niño index is actually about? The index is December–February. Two windows worth
running are the storm season itself, December to March, and the winter-and-after window,
January to September, which is what is left of a calendar year once the disputed autumn is removed.

In [ ]:
def window(site, months, offsets):
    """Mean flow over any set of months, each labelled with the winter year it serves."""
    # 1. Take one month at a time and label it with the winter year it serves. `offsets` is 1 for
    #    a month that runs INTO a winter and 0 for one that follows it; doing this month by month
    #    is what lets a window cross New Year at all.
    daily = flow(site)
    parts = []
    for month, offset in zip(months, offsets):
        part = daily[daily["date"].dt.month == month].copy()
        part["year"] = part["date"].dt.year + offset
        parts.append(part)
    # 2. Stack the relabelled months back up and average within each winter year, so December 1982
    #    and January 1983 fall into the same average.
    mean = pd.concat(parts).groupby("year")["cfs"].mean()
    return pd.DataFrame({"year": mean.index, "cfs": mean.values}).merge(winter, on="year")


for label, months, offsets in [("Dec-Mar, the storm season    ", [12, 1, 2, 3], [1, 0, 0, 0]),
                               ("Jan-Sep, the winter and after", list(range(1, 10)), [0] * 9)]:
    s = window("11152000", months, offsets)
    n = window("11477000", months, offsets)
    r_s, r_n = correlation(s["djf"], s["cfs"]), correlation(n["djf"], n["cfs"])
    print(f"{label}  n = {len(s):3d}   south {r_s:+.3f}   north {r_n:+.3f}   "
          f"gap {r_s - r_n:+.3f}")

## Is the north–south gap real, or two noisy numbers?

The whole claim now rests on the difference between two correlations computed from
77 years each. That is not many, and a correlation from a small sample wanders a long
way. Before quoting the gap, put an interval on it.

**Bootstrap:** Ask the data the same question a thousand times, using a different random slice of itself each time.

**Confidence interval:** Not one number but the range your number would have wandered over, had the world rolled differently.

### ✏️ Your turn 5

Bootstrap each gauge **on its own**, the way you would if you had only one of them.

2000 times over: draw 77 of that gauge's years at random with replacement —
`rng.integers(0, len(south), size=len(south))` gives you their positions — and compute the
correlation of the drawn years. Use `np.random.default_rng(88)` so your run is repeatable.

Report each gauge's 95% interval, and draw the two sets of resampled correlations as histograms on
the same axes so the two intervals are visible at once.

Then print one more line answering it: on these two intervals alone, would you say the two rivers
respond differently to El Niño?

In [ ]:
# ← your answer here



You now have two intervals, and comparing them by eye is the move everybody makes. It is worth
noticing what that move assumes: that the two gauges are two independent studies whose results are
being set side by side. They are not. They are measured over **the same 77 water
years**, against **the same** winter index — when 1983 was a monster El Niño it was a monster El
Niño for both of them.

So there is a second way to resample, and it asks a different question. Instead of asking how far
each correlation could have wandered on its own, resample the **years** once and ask what the
*difference* would have been in that world.

### ✏️ Your turn 6

Bootstrap the **gap**, keeping the two gauges paired.

2000 times over: draw 77 positions with replacement, and use **the same positions
for both gauges** — one `picked` array, two correlations, one subtraction. Collect the
2000 differences.

Report the 95% interval of the difference, its median, and the fraction of the 2000 resamples
in which the southern correlation came out larger than the northern one. Draw the differences as a
histogram with zero marked.

Then print two or three sentences answering it on your own interval: is the north–south gap
established or not, and how does this interval differ from the two you drew in *Your turn 5* — can
a difference be distinguishable from zero even when the two things being differenced are not
distinguishable from each other?

In [ ]:
# ← your answer here



## The question, answered

Not to California — to part of it, and by an amount this notebook cannot pin down. Your own
numbers are above; this is what they add up to.

**The direction is solid.** The southern gauge came out above the northern one under every window
you tried, its own correlation barely moved across all four, and the paired bootstrap in *Your turn
6* put the difference above zero on the same 77 winters. The sign never reversed under
anything this notebook tried. El Niño's grip on California loosens as you go north.

**The size is not solid, and you watched exactly why.** The gap this notebook led with is the
largest of the four you computed, and it is largest because the water year puts the northern
correlation in the low group rather than the high one. Not because the other windows destroyed a
signal — the southern number never moved across any of them — but because a water year hands the
northern river an autumn that runs the other way, and
22% of that river's water is in it. Cut the year at the storm season the
Niño index is actually about, December to March, and the two gauges come out
+0.302 and +0.128, a gap of
+0.174. Take January to September and they are
+0.330 and +0.223, a gap of
+0.108. Nothing in the data prefers one of these; they are answers to
slightly different questions, and the twelve months you call a year decides which question you
asked.

**So the lesson is not "three of four windows agree, so the answer is real."** It is that a
comparison is only as good as the like-for-like it rests on, and here the two rivers were never
being compared on the same thing: the same twelve months mean something different to a river whose
autumn is a fifth of its water than to one whose autumn is a ninth of it. Fix that and the contrast
stays; its magnitude moves by a factor of two. Which of those two things you report is the whole
difference between a finding and a headline.

## What track T5 leans on

**The question.** Does El Niño really bring rain to California?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Baseline** | Write the dumbest rule you can, first. Any model that cannot beat it is decoration. |
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |
| **Table** | A table with a name on every column, so you ask for data by name instead of by position. |
| **NaN** | Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `table.groupby(column)` | split the table into one group per value |
| `column.isna()` | a mask marking where the file had nothing |
| `table.sort_values(by)` | put the rows in order by one column |
| `column.mean()` | the average of a column |
| `pd.to_datetime(column)` | turn a column of date text into real dates, which can be compared and counted |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |
| `rng.integers(low, high, size)` | that many whole numbers drawn at random |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

Two calls in the setup cell are new, and both are plumbing rather than ideas: `np.corrcoef(a, b)` measures how tightly two columns move together (the setup cell wraps it as `correlation`), and `table.merge(other, on="year")` lines two tables up on a shared column.

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track the baseline is the one you built in *Your turn 2* — average the El Niño years,
average the La Niña years, divide. Say what it gives on your gauge, and say what each later step
bought you over it. If a bootstrap interval taught you nothing the ratio had not already told you,
say that too.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so whatever you split, resample or count as
independent has to be split along the structure that is really there — never at random across
rows.

This track fits no model, so there is no train/test split to get wrong. The same idea decided the
answer anyway, in *Your turn 6*: two gauges measured over the same years are not two independent
studies. Name the unit you resampled, say why, and say how much the answer moved when you got it
right.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong, and so does any gauge you
tried that turned out to be a canal.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **WHERE does it switch off, and is the boundary sharp or gradual?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is actually established, and it is less than it looks. Two gauges differ — you have
the difference and its paired interval in your own output above, and you have four other versions
of that difference from *Your turn 3*. Two
gauges are two points. They tell you that the response is not the same everywhere in California and
they tell you nothing whatever about the *shape* of the change between them — whether there is a
line somewhere around the latitude of Monterey with El Niño country on one side of it, or a smooth
ramp running the length of the state, or a patchwork in which what matters is not latitude at all
but which way a catchment faces and how high its snow line sits.

That question needs many gauges, and no one project can fetch many gauges and think carefully about
any of them. **This is why the first section built a list of 192.** Your project takes one — not
one of the two worked here — carries it through everything above, and reports its correlation, its
interval, and its latitude. The class assembles a map that no single project could produce, and the
map is the result.

Four directions, none of them worked out here:

1. **Find the switch.** Take gauges spanning the state, plot each one's correlation against its
   latitude, and look at the shape. Is there a step, and if so where — and how far apart do two
   gauges have to be before their paired difference clears zero?
2. **Test latitude against the alternatives.** The list carries a longitude too, and USGS publishes
   a gauge's elevation and drainage area. A coastal gauge and a Sierra gauge at the same latitude
   receive their water in completely different ways — rain in one, snowmelt in the other — so
   latitude may be standing in for something else entirely. What would distinguish them?
3. **Ask what the record cannot say.** 77 years is 19
   El Niño winters, and your single-gauge interval from *Your turn 5* is wider than the whole
   north–south gap you are trying to resolve. How many gauges would a map need before a boundary
   at, say, 37°N could be told apart from a smooth ramp? Work out what the answer depends on
   before you go looking for it.
4. **Report the window, not just the gap.** Everything above assumes one twelve-month window. Your
   gauge has its own seasonal shape, and the section on the two rival autumns showed that the
   window's cost depends on it — how much of the year's water arrives in October to December. Run
   your gauge under all four windows before you hand the class a number, and hand them the range
   as well as the one you chose. A class map assembled from gauges cut on different conventions,
   or on one convention nobody checked, is a map of the convention.

And one that is bigger than a semester: the 192 gauges in the list are the
314 long records minus the 122 whose *names* advertised
plumbing, and a name is not a measurement. A gauge called
`ARROYO SECO NR SOLEDAD CA` is presumed natural; one with `BL` in its name is presumed regulated and was
dropped. How much of any map you draw is El Niño, and how much of it is California's plumbing?

### ✏️ Your turn 7 — the first move

Before you close this notebook: in a few sentences, what is the **one** measurement you would make
first, what would it show if the change with latitude is a step, what would it show if it is a
ramp, and what number would change your mind? Then make the measurement, in the cell below the
prose.

The gauge list is in `gauges` and `paired` takes any site number in it. Two warnings, and both are
the honest kind. Only the two gauges this notebook worked with have a cached copy stored with the
course, so a gauge you choose yourself is live-only and will fail loudly if USGS is unreachable.
And every gauge you add is another fetch, so choose few and choose them for a reason.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

